In [ ]:
from pyspark.sql.session import SparkSession

def get_spark_session(appName="default"):
    spark = (
        SparkSession
        .builder
        .appName(appName)
        .master("local[*]")
        .config("spark.ui.port","4141")
        .config("spark.port.maxRetries", "0")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark

def stop_start(app_name):
   
    global spark 
    
    if 'spark' in globals():
        try:
            spark.stop()
            print("Old Spark session successfully stopped.")
        except Exception as e:
             print(f"No active session to stop. Error: {e}")
    else:
        print("No existing 'spark' variable found.")

    # 3. Return the fresh session
    print("Starting new Spark session...")
    return get_spark_session(app_name)


spark = stop_start("working_with_different_data_type")

No existing 'spark' variable found.
Starting new Spark session...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/07 13:49:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


##  **Working with Different Types of Data**

1. Booleans
2. Numbers
3. Strings
4. Dates and timestamps
5. Handling null
6. Complex types
7. User-defined functions


DataFrame (Dataset) Methods
This is actually a bit of a trick because a DataFrame is just a Dataset of Row types, so you’ll
actually end up looking at the Dataset methods


[org.apache.spark](!https://spark.apache.org/docs/latest/api/scala/org/index.html)

DataFrameStatFunctions :  holds a variety of statistically related functions, 
DataFrameNaFunctions:  refers to functions that are relevant when working with null data.


In [3]:
flight_df = (
    spark
    .read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load("../data/flight-data/csv/")
)

flight_df.show(5, False)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|United States    |Romania            |1    |
|United States    |Ireland            |264  |
|United States    |India              |69   |
|Egypt            |United States      |24   |
|Equatorial Guinea|United States      |1    |
+-----------------+-------------------+-----+
only showing top 5 rows


In [4]:
flight_df.printSchema()

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)



In [5]:
flight_df.createOrReplaceTempView(name="flight_df")

In [6]:
spark.sql("""
select * from flight_df limit 10
""").show()

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
|    United States|          Singapore|   25|
|    United States|            Grenada|   54|
|       Costa Rica|      United States|  477|
|          Senegal|      United States|   29|
|    United States|   Marshall Islands|   44|
+-----------------+-------------------+-----+



In [7]:
favourite_number = [1,2,3,4]

par = spark.sparkContext.parallelize(favourite_number, 2)

# Wrapping into tuples
test_df = spark.createDataFrame(par.map(lambda x: (x,)), ["number"])
test_df.show()

+------+
|number|
+------+
|     1|
|     2|
|     3|
|     4|
+------+



In [8]:
## Map the RDD elements to Tuples or Rows (Recommended with RDDs)
from pyspark.sql import Row

par_rows = par.map(lambda x: Row(number=x))
test_df = spark.createDataFrame(par_rows)
test_df.show()

## Provide a DDL DataType String

test_df = spark.createDataFrame(par.map(lambda x: (x,)), "number: int")
test_df.show()

+------+
|number|
+------+
|     1|
|     2|
|     3|
|     4|
+------+

+------+
|number|
+------+
|     1|
|     2|
|     3|
|     4|
+------+



In [9]:
### : Skip the RDD and Create the DataFrame Directly

# Direct from local list of tuples
test_df = spark.createDataFrame([(x,) for x in favourite_number], ["number"])
test_df.show()

test_df = par.map(lambda x: (x,)).toDF(["number"])
test_df.show()

+------+
|number|
+------+
|     1|
|     2|
|     3|
|     4|
+------+

+------+
|number|
+------+
|     1|
|     2|
|     3|
|     4|
+------+



In [10]:
text = spark.sparkContext.parallelize([("apple",123,"fruit"),("spinach",60,"vegetable")])

spark.createDataFrame(text).show()

spark.createDataFrame(text, ["edible_name","price_per_kg","type_of_edible"]).show()

+-------+---+---------+
|     _1| _2|       _3|
+-------+---+---------+
|  apple|123|    fruit|
|spinach| 60|vegetable|
+-------+---+---------+

+-----------+------------+--------------+
|edible_name|price_per_kg|type_of_edible|
+-----------+------------+--------------+
|      apple|         123|         fruit|
|    spinach|          60|     vegetable|
+-----------+------------+--------------+



### Converting to Spark Types
- the lit function. This function converts a
type in another language to its correspnding Spark representation

In [11]:
df = spark.createDataFrame(spark.sparkContext.parallelize([(i,) for i in range(5)]), "number:int")

df.show()

+------+
|number|
+------+
|     0|
|     1|
|     2|
|     3|
|     4|
+------+



In [12]:
from pyspark.sql.functions import lit

df.select(lit(5), lit("five"), lit(5.0)).show()

+---+----+---+
|  5|five|5.0|
+---+----+---+
|  5|five|5.0|
|  5|five|5.0|
|  5|five|5.0|
|  5|five|5.0|
|  5|five|5.0|
+---+----+---+



In [13]:
spark.sql(
    """
    select 5 as col1, "five" as col2, 5.0 as col3
    """
).show()

+----+----+----+
|col1|col2|col3|
+----+----+----+
|   5|five| 5.0|
+----+----+----+



### Working with Booleans

- Boolean statements consist of four elements: and, or, true, and false

In [14]:
retail_df = (
    spark
    .read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load("../data/retail-data/")
)

retail_df.show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   537226|    22811|SET OF 6 T-LIGHTS...|       6|2010-12-06 08:34:00|     2.95|   15987.0|United Kingdom|
|   537226|    21713|CITRONELLA CANDLE...|       8|2010-12-06 08:34:00|      2.1|   15987.0|United Kingdom|
|   537226|    22927|GREEN GIANT GARDE...|       2|2010-12-06 08:34:00|     5.95|   15987.0|United Kingdom|
|   537226|    20802|SMALL GLASS SUNDA...|       6|2010-12-06 08:34:00|     1.65|   15987.0|United Kingdom|
|   537226|    22052|VINTAGE CARAVAN G...|      25|2010-12-06 08:34:00|     0.42|   15987.0|United Kingdom|
|   537226|    22705|   WRAP GREEN PEARS |      25|2010-12-06 08:34:00|     0.42|   15987.0|United Kingdom|
|   537226|    20781|GOLD EA

In [15]:
from pyspark.sql.functions import col, instr

filtered_df = (
    retail_df.where(col("InvoiceNo") != 537226)
.select("InvoiceNo", "Description")
)


filtered_df.show(5, False)

+---------+----------------------------------+
|InvoiceNo|Description                       |
+---------+----------------------------------+
|537227   |CHRISTMAS LIGHTS 10 REINDEER      |
|537227   |WICKER WREATH LARGE               |
|537227   |RED DINER WALL CLOCK              |
|537227   |PACK OF 72 RETROSPOT CAKE CASES   |
|537227   |PACK OF 60 PINK PAISLEY CAKE CASES|
+---------+----------------------------------+
only showing top 5 rows


In [16]:
price_filter = col("UnitPrice") > 600
descrptionFilter = instr(col("description"), "POSTAGE") >= 1  ## locate the first occurance of string

# or
descrptionFilter = col("description").contains("POSTAGE") 
filtered_df = (
    retail_df
    .where(price_filter | descrptionFilter)
    .where(col("StockCode").isin("DOT"))
.select("InvoiceNo", "Description")
)


filtered_df.show()

+---------+--------------+
|InvoiceNo|   Description|
+---------+--------------+
|   537237|DOTCOM POSTAGE|
|   537240|DOTCOM POSTAGE|
|   537434|DOTCOM POSTAGE|
|   536544|DOTCOM POSTAGE|
|   536592|DOTCOM POSTAGE|
|   537638|DOTCOM POSTAGE|
|   537641|DOTCOM POSTAGE|
|   537642|DOTCOM POSTAGE|
|   537643|DOTCOM POSTAGE|
|   537645|DOTCOM POSTAGE|
|   537666|DOTCOM POSTAGE|
|   537823|DOTCOM POSTAGE|
|   536862|DOTCOM POSTAGE|
|   536864|DOTCOM POSTAGE|
|   536865|DOTCOM POSTAGE|
|   536876|DOTCOM POSTAGE|
+---------+--------------+



In [17]:
DOTCodeFilter = col("StockCode") == "DOT"
priceFilter = col("UnitPrice") > 600
descripFilter = instr(col("Description"), "POSTAGE") >= 1

retail_df.withColumn("isExpensive", DOTCodeFilter & (priceFilter | descripFilter))\
.where("isExpensive")\
.select("unitPrice", "isExpensive").show(5)

+---------+-----------+
|unitPrice|isExpensive|
+---------+-----------+
|   863.74|       true|
|   940.87|       true|
|   950.99|       true|
|   569.77|       true|
|   607.49|       true|
+---------+-----------+
only showing top 5 rows


In [18]:
retail_df.createOrReplaceTempView("reatil_df_table")

In [19]:
spark.sql("""
          select unitprice, 
          (StockCode="DOT" and ( unitprice > 600 or contains(description,"POSTAGE"))) as isExpensive from reatil_df_table
          where (StockCode="DOT" and (
            unitprice > 600 or contains(description,"POSTAGE"))
          )
         
          """).show()

+---------+-----------+
|unitprice|isExpensive|
+---------+-----------+
|   863.74|       true|
|   940.87|       true|
|   950.99|       true|
|   569.77|       true|
|   607.49|       true|
|   836.14|       true|
|   229.74|       true|
|    448.0|       true|
|   593.15|       true|
|   607.96|       true|
|   701.95|       true|
|   729.42|       true|
|   254.43|       true|
|   121.06|       true|
|   498.47|       true|
|   887.52|       true|
+---------+-----------+



In [20]:
retail_df.withColumn("isExpensive", col("unitprice") >= 200).show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|isExpensive|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+
|   537226|    22811|SET OF 6 T-LIGHTS...|       6|2010-12-06 08:34:00|     2.95|   15987.0|United Kingdom|      false|
|   537226|    21713|CITRONELLA CANDLE...|       8|2010-12-06 08:34:00|      2.1|   15987.0|United Kingdom|      false|
|   537226|    22927|GREEN GIANT GARDE...|       2|2010-12-06 08:34:00|     5.95|   15987.0|United Kingdom|      false|
|   537226|    20802|SMALL GLASS SUNDA...|       6|2010-12-06 08:34:00|     1.65|   15987.0|United Kingdom|      false|
|   537226|    22052|VINTAGE CARAVAN G...|      25|2010-12-06 08:34:00|     0.42|   15987.0|United Kingdom|      false|
|   537226|    22705|   WRAP GREEN PEARS

In [21]:
retail_df.where(col("description").eqNullSafe("STRAWBERRY")).show()

+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
+---------+---------+-----------+--------+-----------+---------+----------+-------+



### Working with Numbers

In [22]:
from pyspark.sql.functions import expr, pow

"""

we were able to multiply our columns together because they were both numerical.
Naturally we can add and subtract as necessary
"""
fabricatedQuantity = pow(col("Quantity") * col("unitprice"), 2) + 5


(
    retail_df
    .withColumn("realQuantity", fabricatedQuantity)
    .select("stockcode","realQuantity")
.show()
)

+---------+------------------+
|stockcode|      realQuantity|
+---------+------------------+
|    22811| 318.2900000000001|
|    21713|            287.24|
|    22927|            146.61|
|    20802|103.00999999999998|
|    22052|            115.25|
|    22705|            115.25|
|    20781|125.56040000000002|
|    22310|103.00999999999998|
|    22389|239.08999999999997|
|    22941|             294.0|
|    22696|            141.89|
|    22193|             294.0|
|    21212|           2545.16|
|    21977| 701.9600000000002|
|    84991| 701.9600000000002|
|    21213| 701.9600000000002|
|    21080|109.03999999999999|
|    22632|10165.640000000003|
|    22315|             230.0|
|    21232|             230.0|
+---------+------------------+
only showing top 20 rows


In [23]:
spark.sql("""show tables""").show()

+---------+---------------+-----------+
|namespace|      tableName|isTemporary|
+---------+---------------+-----------+
|         |      flight_df|       true|
|         |reatil_df_table|       true|
+---------+---------------+-----------+



In [24]:
spark.sql(""" 
select description, power( (quantity * unitprice) , 2) + 5 as realQuantity
 from reatil_df_table
""").show(10,False)

+------------------------------+------------------+
|description                   |realQuantity      |
+------------------------------+------------------+
|SET OF 6 T-LIGHTS CACTI       |318.2900000000001 |
|CITRONELLA CANDLE FLOWERPOT   |287.24            |
|GREEN GIANT GARDEN THERMOMETER|146.61            |
|SMALL GLASS SUNDAE DISH CLEAR |103.00999999999998|
|VINTAGE CARAVAN GIFT WRAP     |115.25            |
|WRAP GREEN PEARS              |115.25            |
|GOLD EAR MUFF HEADPHONES      |125.56040000000002|
|IVORY KNITTED MUG COSY        |103.00999999999998|
|PAPERWEIGHT SAVE THE PLANET   |239.08999999999997|
|CHRISTMAS LIGHTS 10 REINDEER  |294.0             |
+------------------------------+------------------+
only showing top 10 rows


In [25]:
from pyspark.sql.functions import lit, round, bround

df.select(round(lit("2.5")), bround(lit("2.5"))).show(2)

+-------------+--------------+
|round(2.5, 0)|bround(2.5, 0)|
+-------------+--------------+
|          3.0|           2.0|
|          3.0|           2.0|
+-------------+--------------+
only showing top 2 rows


In [26]:
from pyspark.sql.functions import corr

retail_df.select(corr("Quantity", "UnitPrice")).show()

+-------------------------+
|corr(Quantity, UnitPrice)|
+-------------------------+
|     -0.00264404723411...|
+-------------------------+



In [27]:
retail_df.describe().show()

+-------+------------------+------------------+--------------------+-----------------+------------------+------------------+--------------+
|summary|         InvoiceNo|         StockCode|         Description|         Quantity|         UnitPrice|        CustomerID|       Country|
+-------+------------------+------------------+--------------------+-----------------+------------------+------------------+--------------+
|  count|             19632|             19632|               19563|            19632|             19632|             12981|         19632|
|   mean| 537171.9999483897|  28512.7373395967|                NULL|7.549765688671557| 7.718832518337488|15606.855943301749|          NULL|
| stddev|433.12482813623495|18365.746858843257|                NULL|77.43844536458644|203.78479896580612|1769.4463259758816|          NULL|
|    min|            536365|             10002| 4 PURPLE FLOCK D...|            -9360|               0.0|           12347.0|     Australia|
|    max|           

In [28]:
## monotonically_increasing_id. This function generates a unique value for each row
## there are potential determinism issues when doing so.
from pyspark.sql.functions import monotonically_increasing_id

retail_df.select(monotonically_increasing_id(), "*").show(10,False)

+-----------------------------+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+
|monotonically_increasing_id()|InvoiceNo|StockCode|Description                   |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |
+-----------------------------+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+
|0                            |537226   |22811    |SET OF 6 T-LIGHTS CACTI       |6       |2010-12-06 08:34:00|2.95     |15987.0   |United Kingdom|
|1                            |537226   |21713    |CITRONELLA CANDLE FLOWERPOT   |8       |2010-12-06 08:34:00|2.1      |15987.0   |United Kingdom|
|2                            |537226   |22927    |GREEN GIANT GARDEN THERMOMETER|2       |2010-12-06 08:34:00|5.95     |15987.0   |United Kingdom|
|3                            |537226   |20802    |SMALL GLASS SUNDAE DISH CLEAR |6       |2010-12-06 08:34:00|1

## Working with Strings

In [29]:


from pyspark.sql.functions import initcap, instr, lcase, ucase, left, length, locate


description = col("description")

retail_df.select(
                 description, 
                 initcap(description), 
                 lcase(description),
                 ucase(description),
                #  len(description), not in pyspark
                 length(description),
                 locate("GLASS",description)
                ).show(10,False)

+------------------------------+------------------------------+------------------------------+------------------------------+-------------------+-----------------------------+
|description                   |initcap(description)          |lcase(description)            |ucase(description)            |length(description)|locate(GLASS, description, 1)|
+------------------------------+------------------------------+------------------------------+------------------------------+-------------------+-----------------------------+
|SET OF 6 T-LIGHTS CACTI       |Set Of 6 T-lights Cacti       |set of 6 t-lights cacti       |SET OF 6 T-LIGHTS CACTI       |24                 |0                            |
|CITRONELLA CANDLE FLOWERPOT   |Citronella Candle Flowerpot   |citronella candle flowerpot   |CITRONELLA CANDLE FLOWERPOT   |27                 |0                            |
|GREEN GIANT GARDEN THERMOMETER|Green Giant Garden Thermometer|green giant garden thermometer|GREEN GIANT GARDEN THERMOM

In [30]:
from pyspark.sql.functions import lower, lpad, rpad, ltrim, rtrim, mask, overlay

description = col("description")

retail_df.select(
                 description, 
                  lpad(description, 20 , "*"),
                  rpad(description, 20, "*"),
                  ltrim(description),
                  rtrim(description),
                  mask(description, lit("X"), lit("L"), lit("7")),
                  overlay(description, lit(" replace "), lit("2")) # replace from position

                ).show(10,False)

+------------------------------+------------------------+------------------------+------------------------------+------------------------------+--------------------------------+--------------------------------------+
|description                   |lpad(description, 20, *)|rpad(description, 20, *)|ltrim(description)            |rtrim(description)            |mask(description, X, L, 7, NULL)|overlay(description,  replace , 2, -1)|
+------------------------------+------------------------+------------------------+------------------------------+------------------------------+--------------------------------+--------------------------------------+
|SET OF 6 T-LIGHTS CACTI       |SET OF 6 T-LIGHTS CA    |SET OF 6 T-LIGHTS CA    |SET OF 6 T-LIGHTS CACTI       |SET OF 6 T-LIGHTS CACTI       |XXX XX 7 X-XXXXXX XXXXX         |S replace -LIGHTS CACTI               |
|CITRONELLA CANDLE FLOWERPOT   |CITRONELLA CANDLE FL    |CITRONELLA CANDLE FL    |CITRONELLA CANDLE FLOWERPOT   |CITRONELLA CANDLE F

In [31]:


from pyspark.sql.functions import position

retail_df.select(
    position(lit("GLASS"), description)
).show(10,False)

+-------------------------------+
|position(GLASS, description, 1)|
+-------------------------------+
|0                              |
|0                              |
|0                              |
|7                              |
|0                              |
|0                              |
|0                              |
|0                              |
|0                              |
|0                              |
+-------------------------------+
only showing top 10 rows


In [32]:
df.select(
lpad(lit("HELLO"), 3, " ").alias("lp"),
rpad(lit("HELLO"), 10, " ").alias("rp")).show(2)

+---+----------+
| lp|        rp|
+---+----------+
|HEL|HELLO     |
|HEL|HELLO     |
+---+----------+
only showing top 2 rows


### Regular Expressions

In [33]:
retail_df = (
    spark
    .read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load("../data/retail-data/")
)


In [34]:
from pyspark.sql.functions import regexp_replace, regexp_extract, translate

lst = ["black", "white", "red", "green", "blue"]
lst_upper = map(lambda x: x.upper(), lst)

regex_string = "|".join(lst_upper)

print(regex_string)

description = col("description")

retail_df.select(
 description,
regexp_replace(description, pattern=regex_string, replacement="COLOR"),
translate(descripFilter, "LEET", "XXX")

).show(4, False)

BLACK|WHITE|RED|GREEN|BLUE
+------------------------------+-----------------------------------------------------------------+--------------------------------------------------------+
|description                   |regexp_replace(description, BLACK|WHITE|RED|GREEN|BLUE, COLOR, 1)|translate((instr(Description, POSTAGE) >= 1), LEET, XXX)|
+------------------------------+-----------------------------------------------------------------+--------------------------------------------------------+
|SET OF 6 T-LIGHTS CACTI       |SET OF 6 T-LIGHTS CACTI                                          |false                                                   |
|CITRONELLA CANDLE FLOWERPOT   |CITRONELLA CANDLE FLOWERPOT                                      |false                                                   |
|GREEN GIANT GARDEN THERMOMETER|COLOR GIANT GARDEN THERMOMETER                                   |false                                                   |
|SMALL GLASS SUNDAE DISH CLEAR |SMALL

In [38]:
spark.sql("show tables").show()

+---------+---------------+-----------+
|namespace|      tableName|isTemporary|
+---------+---------------+-----------+
|         |      flight_df|       true|
|         |reatil_df_table|       true|
+---------+---------------+-----------+



In [40]:
spark.sql("""
          
         select description, translate(description, "LEET", "1337") as translated_description FROM reatil_df_table
          """).show()

+--------------------+----------------------+
|         description|translated_description|
+--------------------+----------------------+
|SET OF 6 T-LIGHTS...|  S37 OF 6 7-1IGH7S...|
|CITRONELLA CANDLE...|  CI7RON311A CAND13...|
|GREEN GIANT GARDE...|  GR33N GIAN7 GARD3...|
|SMALL GLASS SUNDA...|  SMA11 G1ASS SUNDA...|
|VINTAGE CARAVAN G...|  VIN7AG3 CARAVAN G...|
|   WRAP GREEN PEARS |     WRAP GR33N P3ARS |
|GOLD EAR MUFF HEA...|  GO1D 3AR MUFF H3A...|
|IVORY KNITTED MUG...|  IVORY KNI773D MUG...|
|PAPERWEIGHT SAVE ...|  PAP3RW3IGH7 SAV3 ...|
|CHRISTMAS LIGHTS ...|  CHRIS7MAS 1IGH7S ...|
| WICKER WREATH LARGE|   WICK3R WR3A7H 1ARG3|
|RED DINER WALL CLOCK|  R3D DIN3R WA11 C1OCK|
|PACK OF 72 RETROS...|  PACK OF 72 R37ROS...|
|PACK OF 60 PINK P...|  PACK OF 60 PINK P...|
|60 TEATIME FAIRY ...|  60 73A7IM3 FAIRY ...|
|PACK OF 72 SKULL ...|  PACK OF 72 SKU11 ...|
|SET/20 RED RETROS...|  S37/20 R3D R37ROS...|
|HAND WARMER RED R...|  HAND WARM3R R3D R...|
|200 RED + WHITE B...|  200 R3D + 

In [41]:
from  pyspark.sql.functions import contains


contains_black = col("description").contains("BLACK")
contains_white = col("description").contains("WHITE")

(

    retail_df
    .withColumn("hasSapmleColor",contains_black | contains_white)
    .where("hasSapmleColor")
    .show(10,False)
)

+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+--------------+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |hasSapmleColor|
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+--------------+
|537227   |22315    |200 RED + WHITE BENDY STRAWS       |12      |2010-12-06 08:42:00|1.25     |17677.0   |United Kingdom|true          |
|537227   |85014A   |BLACK/BLUE POLKADOT UMBRELLA       |3       |2010-12-06 08:42:00|5.95     |17677.0   |United Kingdom|true          |
|537233   |82484    |WOOD BLACK BOARD ANT WHITE FINISH  |12      |2010-12-06 09:37:00|5.55     |17428.0   |United Kingdom|true          |
|537237   |20727    |LUNCH BAG  BLACK SKULL.            |1       |2010-12-06 09:58:00|4.21     |NULL      |United Kingdom|true          |
|537237   |20735    |BLACK MINI TA

In [50]:
 ### This is trivial with just two values, but it becomes more complicated when there are values.



color_lst = ['BLACK','WHITE','ORANGE','RED']

column_filter = list()
df = retail_df
for col_name in color_lst:
    df = retail_df.withColumn(col_name, (col("description").contains(col_name)))


df.where(col("red") | col("black")).show(10,False)


+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+-----+-----+------+-----+
|InvoiceNo|StockCode|Description                        |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |BLACK|WHITE|ORANGE|RED  |
+---------+---------+-----------------------------------+--------+-------------------+---------+----------+--------------+-----+-----+------+-----+
|537227   |22193    |RED DINER WALL CLOCK               |2       |2010-12-06 08:42:00|8.5      |17677.0   |United Kingdom|false|false|false |true |
|537227   |21080    |SET/20 RED RETROSPOT PAPER NAPKINS |12      |2010-12-06 08:42:00|0.85     |17677.0   |United Kingdom|false|false|false |true |
|537227   |22632    |HAND WARMER RED RETROSPOT          |48      |2010-12-06 08:42:00|2.1      |17677.0   |United Kingdom|false|false|false |true |
|537227   |22315    |200 RED + WHITE BENDY STRAWS       |12      |2010-12-06 08:42:00|1.25     |17677.0   |Unite

In [54]:
locator = [locate(color_name, col("description")  ).cast("boolean").alias(f"is_{color_name.lower()}") for color_name in color_lst  ]

retail_df.select(*locator, "*").show(10, False)

+--------+--------+---------+------+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+-----+-----+------+-----+
|is_black|is_white|is_orange|is_red|InvoiceNo|StockCode|Description                   |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |BLACK|WHITE|ORANGE|RED  |
+--------+--------+---------+------+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+-----+-----+------+-----+
|false   |false   |false    |false |537226   |22811    |SET OF 6 T-LIGHTS CACTI       |6       |2010-12-06 08:34:00|2.95     |15987.0   |United Kingdom|false|false|false |false|
|false   |false   |false    |false |537226   |21713    |CITRONELLA CANDLE FLOWERPOT   |8       |2010-12-06 08:34:00|2.1      |15987.0   |United Kingdom|false|false|false |false|
|false   |false   |false    |false |537226   |22927    |GREEN GIANT GARDEN THERMOMETER|2       |2010-12-06 08:

In [55]:
spark.stop()